# Audio Hybrid Neuroevolution — Concurrent Individuals + Rotating Fold (`neo_conc`)

Mejora sobre `test.ipynb` usando el paquete **`neo_conc`** (recreado desde cero a partir de `neuroevolution`).

## Cambios clave

| Aspecto | `test.ipynb` (paquete `neuroevolution`) | **Este notebook (`neo_conc`)** |
|---|---|---|
| Folds por individuo y generación | 5 folds en paralelo (1 thread por fold) | **1 fold por generación**, rotado |
| Individuos por batch | 1 individuo a la vez (5 folds simultáneos) | **5 individuos en paralelo** sobre el mismo fold |
| Pattern de paralelismo | `ThreadPoolExecutor` × folds | `ThreadPoolExecutor` × individuos + `torch.cuda.Stream()` (estilo `multi_cnn_single_gpu_parallel_training.ipynb`) |
| Riesgo de overfit a un fold | Bajo (siempre ve los 5) | **Mitigado por rotación** (cada generación usa un fold distinto) |
| Coste por generación | `pop_size × 5` entrenamientos | `pop_size × 1` entrenamientos |

## Esquema de evaluación

```
Generation g  →  fold = ((g + offset - 1) mod num_folds) + 1
Population (N individuos)
        │
        ▼
  Batches de tamaño `concurrent_individuals` (def: 5)
        │   ┌────────────┐  ┌────────────┐  ┌────────────┐
        └─▶ │ stream 0   │  │ stream 1   │  │ stream 2   │ ... (misma GPU)
            │ genoma A   │  │ genoma B   │  │ genoma C   │
            │ AMP fp16   │  │ AMP fp16   │  │ AMP fp16   │
            └────────────┘  └────────────┘  └────────────┘
                              fold compartido (cargado 1 vez)
```

Al final, el mejor individuo se vuelve a evaluar con el módulo `evaluation.cross_validation` heredado (5-fold completo) para obtener métricas científicas comparables con los otros notebooks.

## 1. Setup del entorno

In [ ]:
from neo_conc import verify_dependencies, REQUIRED_PACKAGES

verify_dependencies(REQUIRED_PACKAGES)

In [ ]:
import os
from datetime import datetime

from neo_conc import (
    CONFIG,
    setup_device,
    setup_seeds,
    setup_logging,
    load_dataset,
    HybridNeuroevolution,
    plot_fitness_evolution,
    show_evolution_statistics,
    analyze_failed_evaluations,
    configure_plot_style,
    evaluate_5fold_cross_validation,
)

print('neo_conc loaded OK')

## 2. Configuración

Los parámetros nuevos relevantes son:

* `concurrent_individuals` — cuántos individuos se entrenan a la vez en la misma GPU (default 5).
* `fold_rotation_strategy` — `'sequential'` (rota fold 1→2→...→5→1) o `'random'`.
* `fold_rotation_offset` — fold inicial (1 por defecto).

In [ ]:
# Rutas de datos y artefactos
CONFIG['data_path'] = os.path.join('data', 'sets', 'folds_5')
CONFIG['artifacts_dir'] = os.path.join('artifacts', 'neo_conc_audio')
CONFIG['artifact_dir'] = CONFIG['artifacts_dir']

# Concurrencia: 5 individuos a la vez, fold rotando cada generación
CONFIG['concurrent_individuals'] = 5
CONFIG['fold_rotation_strategy'] = 'sequential'
CONFIG['fold_rotation_offset'] = 1

# Tamaño de población múltiplo de concurrent_individuals para batches limpios
CONFIG['population_size'] = 20
CONFIG['max_generations'] = 100
CONFIG['fitness_threshold'] = 95.0

# Setup
device = setup_device()
setup_seeds(42)
log_path = os.path.join(CONFIG['artifacts_dir'], 'execution_log.txt')
setup_logging(log_path)
configure_plot_style()

print('=' * 60)
print('NEO_CONC — CONFIGURATION')
print('=' * 60)
print(f"Population size           : {CONFIG['population_size']}")
print(f"Concurrent individuals    : {CONFIG['concurrent_individuals']}  (per GPU batch)")
print(f"Folds available           : {CONFIG['num_folds']}")
print(f"Fold rotation strategy    : {CONFIG['fold_rotation_strategy']}")
print(f"Max generations           : {CONFIG['max_generations']}")
print(f"Fitness threshold         : {CONFIG['fitness_threshold']}%")
print(f"Device                    : {device}")
print(f"Artifacts dir             : {CONFIG['artifacts_dir']}")
print('=' * 60)

## 3. Verificar dataset

In [ ]:
load_dataset(CONFIG)

## 4. Ejecución de la neuroevolución

Para cada generación `g`:

1. Se selecciona el fold `f = ((g + offset - 1) mod num_folds) + 1`.
2. La población se trocea en batches de `concurrent_individuals`.
3. Cada batch entrena en paralelo en la misma GPU usando `torch.cuda.Stream()` + AMP.
4. Fitness por individuo = F1-score en el fold `f` (sin promedio entre folds).

La rotación evita que la población se ajuste a las particularidades de un único fold.

In [ ]:
start_time = datetime.now()
print(f"\nStarting neo_conc neuroevolution at {start_time.strftime('%H:%M:%S')}")
print(f"Architecture: Conv1D -> BatchNorm1D -> Activation -> MaxPool1D -> FC")
print(f"Each generation evaluates the population on a SINGLE rotated fold")
print(f"Up to {CONFIG['concurrent_individuals']} individuals train concurrently per GPU batch")
print('=' * 60)

neuroevolution = HybridNeuroevolution(CONFIG, device)
best_genome = neuroevolution.evolve()

end_time = datetime.now()
execution_time = end_time - start_time

print(f"\n{'=' * 60}")
print('EVOLUTION PROCESS COMPLETED')
print('=' * 60)
print(f"Completed at         : {end_time.strftime('%H:%M:%S')}")
print(f"Total execution time : {execution_time}")
print(f"Generations processed: {neuroevolution.generation}")
print(f"Best fitness         : {best_genome['fitness']:.2f}%")
print(f"Progress JSON        : {neuroevolution.progress_json_path}")
print(f"Generation log       : {neuroevolution.generation_log_path}")
print('=' * 60)

## 5. Inspección del esquema de rotación de folds

Comprobamos qué fold tocó en cada generación, para que sea evidente que el sistema no se quedó pegado a uno solo.

In [ ]:
from collections import Counter

fold_history = neuroevolution.fold_history
print(f"Total generations evaluated: {len(fold_history)}")
print(f"\nFold per generation:")
for entry in fold_history:
    print(f"  gen {entry['generation']:>3} -> fold {entry['fold']}")

counts = Counter(e['fold'] for e in fold_history)
print(f"\nFold usage distribution:")
for fold_num in sorted(counts):
    print(f"  fold {fold_num}: {counts[fold_num]} generation(s)")

## 6. Visualización de la evolución

In [ ]:
plot_fitness_evolution(neuroevolution, CONFIG)

In [ ]:
show_evolution_statistics(neuroevolution, CONFIG)

In [ ]:
analyze_failed_evaluations(neuroevolution)

## 7. Mejor arquitectura encontrada

In [ ]:
print('\n' + '=' * 80)
print('BEST ARCHITECTURE DETAILS')
print('=' * 80 + '\n')
best = neuroevolution.best_individual
print(f"ID                : {best['id']}")
print(f"Fitness (1 fold)  : {best['fitness']:.2f}%")
print(f"Evaluation fold   : {best.get('evaluation_fold', 'n/a')}")
print(f"Conv layers       : {best['num_conv_layers']}")
print(f"FC layers         : {best['num_fc_layers']}")
print(f"Optimizer         : {best['optimizer']}")
print(f"Learning rate     : {best['learning_rate']}")
print(f"Filters per layer : {best.get('filters_per_layer')}")
print(f"Kernel sizes      : {best.get('kernel_sizes_per_layer')}")
print(f"FC nodes          : {best.get('fc_nodes_per_layer')}")
print(f"Dropout           : {best.get('dropout_rate')}")

## 8. Evaluación final 5-fold del mejor individuo

Como cada generación sólo vio un fold, hacemos una evaluación 5-fold completa **solo del mejor genoma** para obtener las métricas comparables con los otros notebooks (Accuracy, Sensitivity, Specificity, Precision, F1, AUC con std).

In [ ]:
final_metrics = evaluate_5fold_cross_validation(
    best_genome,
    CONFIG,
    device,
)

print('\n' + '#' * 80)
print('FINAL 5-FOLD EVALUATION OF THE BEST GENOME')
print('#' * 80)
for key in ('accuracy', 'sensitivity', 'specificity', 'precision', 'f1_score', 'auc'):
    mean = final_metrics.get(key, 0.0)
    std = final_metrics.get(f'{key}_std', 0.0)
    print(f"  {key:<12}: {mean:>6.2f}% ± {std:>5.2f}%")

## 9. Cargar el mejor checkpoint (opcional)

In [ ]:
loaded_genome, loaded_model = neuroevolution.load_best_checkpoint()
if loaded_model is not None:
    print(f"Checkpoint OK | id={loaded_genome['id']} | fitness={loaded_genome['fitness']:.2f}%")

## Resumen

* La carpeta `neo_conc/` es independiente (todos los imports son internos al paquete).
* Cambia el eje de paralelismo: ahora son **N individuos × 1 fold** por generación, en vez de **1 individuo × 5 folds**.
* La rotación de fold por generación reduce el riesgo de sobreajuste al split.
* El coste por generación baja a `pop_size × 1` entrenamientos (frente a `pop_size × 5`), lo que permite poblaciones mayores o más generaciones.
* La evaluación final 5-fold del mejor individuo se mantiene para reportar métricas comparables.